In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIGURES_DIR = PROJECT_ROOT / "figures"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdm_learn.modeling import LOOCV_grouped_plot, area_table, heatmap, ks_pvalue
from pdm_learn.oncogene import (
    ONCOGENE_HEATMAP_DIMENSIONS,
    evaluate_method_curves,
    load_oncogene_feature_sets,
    plot_method_curves,
    rank_candidate_oncogenes,
)


In [ ]:
data_dict, input_data = load_oncogene_feature_sets(DATA_DIR)
positive, negative = data_dict["PDM"]

pd.DataFrame(
    {
        "dataset": list(data_dict.keys()),
        "positive_genes": [len(value[0]) for value in data_dict.values()],
        "negative_genes": [len(value[1]) for value in data_dict.values()],
    }
)


In [ ]:
input_data.head()


In [ ]:
MODEL_LIST = ("SVR", "XGB", "GBR", "MLP", "LR", "GNB")
DEFAULT_MODEL = "XGB"
KS_TEST_BY_METHOD = {"PDM": True}

MODEL_LIST, KS_TEST_BY_METHOD


In [ ]:
ranking = rank_candidate_oncogenes(
    positive,
    negative,
    trials=100,
    model=DEFAULT_MODEL,
    ks_test=True,
)
ranking.head(20)


In [ ]:
LOOCV_grouped_plot(
    data_dict,
    5,
    models=MODEL_LIST,
    ks_test=KS_TEST_BY_METHOD,
)


In [ ]:
pr_results = evaluate_method_curves(
    data_dict,
    trials=25,
    model=DEFAULT_MODEL,
    ks_test=KS_TEST_BY_METHOD,
    metric="pr",
)
plot_method_curves(
    pr_results,
    title="Oncogene Precision-Recall Curves",
    xlabel="Recall",
    ylabel="Precision",
)
pr_results


In [ ]:
loocv_results = evaluate_method_curves(
    data_dict,
    trials=25,
    model=DEFAULT_MODEL,
    ks_test=KS_TEST_BY_METHOD,
    metric="loocv",
)
plot_method_curves(
    loocv_results,
    title="Oncogene LOOCV Curves",
    xlabel="Rank",
    ylabel="Cumulative positives",
)
loocv_results


In [ ]:
feature_arr = [10, 20, 50, 100, 124, 150, 200, 250, 300, 349]
areas = area_table(
    positive.iloc[:, 1:].to_numpy(),
    negative.iloc[:, 1:].to_numpy(),
    20,
    model=DEFAULT_MODEL,
    feat_arr=feature_arr,
)

plt.figure(figsize=(8, 4))
plt.plot(feature_arr, areas, marker="o")
plt.title("PDM Feature Sweep")
plt.xlabel("Selected Features")
plt.ylabel("LOOCV Area")
plt.tight_layout()
plt.show()


In [ ]:
p_val = ks_pvalue(positive.iloc[:, 1:].to_numpy(), negative.iloc[:, 1:].to_numpy())
log_p_val = np.log10(np.clip(p_val, 1e-300, None))
heatmap(
    log_p_val,
    ONCOGENE_HEATMAP_DIMENSIONS,
    cmap="gist_heat_r",
    min=float(np.min(log_p_val)),
    max=float(np.max(log_p_val)),
    flip=True,
    axes=False,
    colorbar=False,
)
